In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent:
    if (root / 'utils.py').exists():
        break
    root = root.parent

if str(root) not in sys.path:
    sys.path.append(str(root))

In [ ]:
import pandas as pd
from utils import cfg

In [ ]:
application = pd.read_csv(cfg.get_modelling_train('application'))
bureau = pd.read_csv(cfg.get_modelling_train('bureau'))
bureau_balance = pd.read_csv(cfg.get_modelling_train('bureau_balance'))
prev_application = pd.read_csv(cfg.get_modelling_train('previous_application'))
credit_card_balance = pd.read_csv(cfg.get_modelling_train('credit_card_balance'))
installments_payments = pd.read_csv(cfg.get_modelling_train('installments_payments'))
pos_cash_balance = pd.read_csv(cfg.get_modelling_train('pos_cash_balance'))

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from srcs.pipelines.data_pipeline import DataPipeline, RawDataBundle, DataPipelineConfig

In [ ]:
RAW_DIR = root
OUTPUT_DIR = RAW_DIR / "Home Credit Dataset" / "cv" 

N_SPLITS = 5
RANDOM_STATE = 42

def filter_raw_tables(tables, ids):
    ids = set(ids)

    tables["application"] = tables["application"][tables["application"]["SK_ID_CURR"].isin(ids)]
    tables["prev_application"] = tables["prev_application"][tables["prev_application"]["SK_ID_CURR"].isin(ids)]
    tables["bureau"] = tables["bureau"][tables["bureau"]["SK_ID_CURR"].isin(ids)]

    bureau_ids = tables["bureau"]["SK_ID_BUREAU"]
    tables["bureau_balance"] = tables["bureau_balance"][tables["bureau_balance"]["SK_ID_BUREAU"].isin(bureau_ids)]
    tables["credit_card_balance"] = tables["credit_card_balance"][tables["credit_card_balance"]["SK_ID_CURR"].isin(ids)]

    tables["pos_cash_balance"] = tables["pos_cash_balance"][tables["pos_cash_balance"]["SK_ID_CURR"].isin(ids)]

    tables["installments_payments"] = tables["installments_payments"][tables["installments_payments"]["SK_ID_CURR"].isin(ids)]

    return tables

tables = {
    "application": application,
    "prev_application": prev_application,
    "bureau": bureau,
    "bureau_balance": bureau_balance,
    "credit_card_balance": credit_card_balance,
    "pos_cash_balance": pos_cash_balance,
    "installments_payments": installments_payments,
}

def build_pipeline(table) -> pd.DataFrame:
    config = DataPipelineConfig(target_col="TARGET", reduce_memory=True)
    raw = RawDataBundle(
        application=table['application'],
        prev_application=table['prev_application'],
        bureau=table['bureau'],
        bureau_balance=table['bureau_balance'],
        credit_card_balance=table['credit_card_balance'],
        pos_cash_balance=table['pos_cash_balance'],
        instalments_payments=table['installments_payments']
    )

    _, _, dataset = DataPipeline(config=config).run(raw)
    return dataset

application = tables["application"]
X = application[["SK_ID_CURR"]]
y = application["TARGET"]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    train_ids = X.iloc[train_idx]["SK_ID_CURR"]
    valid_ids = X.iloc[valid_idx]["SK_ID_CURR"]

    train_tables = {name: df.copy() for name, df in tables.items()}
    valid_tables = {name: df.copy() for name, df in tables.items()}

    train_tables = filter_raw_tables(train_tables, train_ids)
    valid_tables = filter_raw_tables(valid_tables, valid_ids)

    train_df = build_pipeline(train_tables)
    valid_df = build_pipeline(valid_tables)

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_df.to_csv(fold_dir / f"train_{fold}.csv", index=False)
    valid_df.to_csv(fold_dir / f"valid_{fold}.csv",index=False)

    print(
        f"Fold {fold}: "
        f"train IDs={len(train_ids)}, "
        f"valid IDs={len(valid_ids)}, "
        f"train shape={train_df.shape}, "
        f"valid shape={valid_df.shape}"
    )

